In [ ]:
from docx import Document
from pathlib import Path
import re
import json


# CONFIG

ROOT_DIR = Path(
    r"C:\Users\ASUS\Desktop\3gpp_specs"
)

OUTPUT_FILE = ROOT_DIR / "parsed_sections.json"


# HELPERS

def get_heading_level(style):
    """
    Convert:

        Heading 1 -> 1
        Heading 2 -> 2
        Heading 3 -> 3

    Returns None if the style is not a heading.
    """

    if style.startswith("Heading "):
        try:
            return int(style.split()[-1])
        except ValueError:
            return None

    return None


def parse_section_heading(text):
    """
    Extract section number and title.

    Examples:

        4.2.3 Service Request procedures
        -> ('4.2.3', 'Service Request procedures')

        4.2.3.1 General
        -> ('4.2.3.1', 'General')
    """

    pattern = r"^(\d+(?:\.\d+)*[A-Za-z]?)\s+(.+)$"

    match = re.match(pattern, text.strip())

    if match:
        section_number = match.group(1)
        title = match.group(2).strip()

        return section_number, title

    return None, text.strip()


def clean_text(text):
    """
    Basic whitespace cleanup.
    """

    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# PARSE ONE DOCX

def parse_3gpp_docx(path):
    """
    Parse one DOCX file into structured sections.
    """

    doc = Document(path)

    sections = []

    current_section = None

    # Keeps the latest heading at each heading level
    heading_stack = {}

    # Try to determine specification name from folder/file


    spec = None

    for part in path.parts:

        if re.fullmatch(r"\d{2}\.\d{3}", part):
            spec = f"TS {part}"
            break

    # Fallback if folder structure is different
    if spec is None:

        match = re.search(
            r"(\d{5})",
            path.stem
        )

        if match:

            number = match.group(1)

            spec = (
                f"TS {number[:2]}."
                f"{number[2:]}"
            )

        else:
            spec = path.stem

    # Process paragraphs

    for paragraph in doc.paragraphs:

        text = clean_text(paragraph.text)
        style = paragraph.style.name.strip()

        # Ignore empty paragraphs
        if not text:
            continue

        # Ignore Table of Contents

        if style.lower().startswith("toc"):
            continue

        
        # Check if this is a heading

        heading_level = get_heading_level(style)

        # HEADING

        if heading_level is not None:

            section_number, title = parse_section_heading(text)

            if section_number is None:
                section_number = ""

            # Save previous section

            if current_section is not None:

                current_section["content"] = "\n".join(
                    current_section["content"]
                ).strip()

                sections.append(current_section)

            # Remove deeper/current headings

            for level in list(heading_stack.keys()):

                if level >= heading_level:
                    del heading_stack[level]

            # Store current heading

            heading_stack[heading_level] = {
                "section": section_number,
                "title": title
            }

            # Find parent

            parent = None

            parent_levels = [
                level
                for level in heading_stack
                if level < heading_level
            ]

            if parent_levels:

                parent_level = max(parent_levels)

                parent = heading_stack[parent_level]

            # Create section

            current_section = {

                # Document metadata
                "spec": spec,
                "source": path.name,

                # Section metadata
                "section": section_number,
                "title": title,
                "heading_level": heading_level,

                # Parent metadata
                "parent_section": (
                    parent["section"]
                    if parent
                    else None
                ),

                "parent_title": (
                    parent["title"]
                    if parent
                    else None
                ),

                # Content
                "content": []
            }

        # BODY CONTENT

        else:

            if current_section is not None:

                current_section["content"].append(text)

    # SAVE LAST SECTION

    if current_section is not None:

        current_section["content"] = "\n".join(
            current_section["content"]
        ).strip()

        sections.append(current_section)

    return sections


# FIND ALL DOCX FILES

docx_files = list(
    ROOT_DIR.rglob("*.docx")
)

print("=" * 80)
print("3GPP DOCX FILES FOUND")
print("=" * 80)

print(f"Total DOCX files: {len(docx_files)}")

for path in docx_files:
    print(path)


# PARSE ALL FILES


all_sections = []

print("\n" + "=" * 80)
print("PARSING FILES")
print("=" * 80)


for i, path in enumerate(docx_files, start=1):

    print(
        f"\n[{i}/{len(docx_files)}] "
        f"Parsing: {path.name}"
    )

    try:

        sections = parse_3gpp_docx(path)

        all_sections.extend(sections)

        print(
            f"    Sections extracted: "
            f"{len(sections)}"
        )

    except Exception as e:

        print(
            f"    ERROR: {e}"
        )


# SAVE EVERYTHING INTO ONE JSON FILE

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_sections,
        f,
        ensure_ascii=False,
        indent=2
    )


# SUMMARY

print("\n" + "=" * 80)
print("EXTRACTION COMPLETE")
print("=" * 80)

print(
    f"Files processed : {len(docx_files)}"
)

print(
    f"Total sections  : {len(all_sections)}"
)

print(
    f"Output file     : {OUTPUT_FILE}"
)

3GPP DOCX FILES FOUND
Total DOCX files: 14
C:\Users\ASUS\Desktop\3gpp_specs\23.288\23288-id0.docx
C:\Users\ASUS\Desktop\3gpp_specs\23.501\23501-j80\23501-j80.docx
C:\Users\ASUS\Desktop\3gpp_specs\23.501\23501-j80\~$501-j80.docx
C:\Users\ASUS\Desktop\3gpp_specs\23.502\23502-k20\23502-k20.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_0_cover.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_1_Main-Body_s00_s04.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_2_Main-Body_s05_s0504.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_3_Main-Body_s0505_s0506.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_4_Main-Body_s06_s08.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_5_Main-Body_s09_s10.docx
C:\Users\ASUS\Desktop\3gpp_specs\24.501\24501-k00\24501-k00_6_Annexes_sA_sHistory.docx
C:\Users\ASUS\Desktop\3gpp_specs\33.501\33501-j70\33501-j70.docx
C:\Users\ASUS\Desktop\3gpp_specs\38.300\38300-fn0\3

In [ ]:
from pathlib import Path
import json

from langchain_text_splitters import RecursiveCharacterTextSplitter


# 1. PATHS

ROOT_DIR = Path(
    r"C:\Users\ASUS\Desktop\3gpp_specs"
)

INPUT_FILE = ROOT_DIR / "parsed_sections.json"
OUTPUT_FILE = ROOT_DIR / "parsed_chunks.json"


# 2. LOAD PARSED SECTIONS

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    sections = json.load(f)

print("=" * 80)
print("LOADED PARSED SECTIONS")
print("=" * 80)

print(f"Total sections: {len(sections)}")


# 3. CREATE CHUNKER

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


# 4. CHUNK EACH SECTION

all_chunks = []


for section_index, section in enumerate(sections):

    text = section["content"].strip()

    # Skip empty sections
    if not text:
        continue

    # Split ONLY this section

    chunks = splitter.split_text(text)

    # Create a document for every chunk

    for chunk_index, chunk_text in enumerate(chunks):


        chunk_id = (
        f"{section['spec'].replace(' ', '_')}_"
        f"section_{section_index}_"
        f"chunk_{chunk_index:03d}"
                    )
        

        chunk = {
            # Original metadata
            "spec": section["spec"],
            "source": section["source"],
            "section": section["section"],
            "title": section["title"],
            "heading_level": section["heading_level"],
            "parent_section": section["parent_section"],
            "parent_title": section["parent_title"],

            # New chunk metadata
            "chunk_id": chunk_id,
            "chunk_index": chunk_index,

            # Actual text
            "text": chunk_text
        }

        all_chunks.append(chunk)


# 5. SAVE CHUNKS

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )



# 6. SHOW RESULTS

print("\n" + "=" * 80)
print("CHUNKING COMPLETE")
print("=" * 80)

print(f"Sections processed : {len(sections)}")
print(f"Total chunks       : {len(all_chunks)}")
print(f"Output file        : {OUTPUT_FILE}")


# 7. PREVIEW FIRST 5 CHUNKS

print("\n" + "=" * 80)
print("FIRST 5 CHUNKS")
print("=" * 80)

for chunk in all_chunks[:5]:

    print("\n" + "-" * 80)

    print(f"Chunk ID       : {chunk['chunk_id']}")
    print(f"Spec           : {chunk['spec']}")
    print(f"Section        : {chunk['section']}")
    print(f"Title          : {chunk['title']}")
    print(f"Parent         : {chunk['parent_section']} - "
          f"{chunk['parent_title']}")
    print(f"Chunk index    : {chunk['chunk_index']}")
    print(f"Characters     : {len(chunk['text'])}")

    print("\nText:")
    print(chunk["text"][:500])

LOADED PARSED SECTIONS
Total sections: 6302

CHUNKING COMPLETE
Sections processed : 6302
Total chunks       : 15773
Output file        : C:\Users\ASUS\Desktop\3gpp_specs\parsed_chunks.json

FIRST 5 CHUNKS

--------------------------------------------------------------------------------
Chunk ID       : TS_23.288_section_0_chunk_000
Spec           : TS 23.288
Section        : 
Title          : Foreword
Parent         : None - None
Chunk index    : 0
Characters     : 805

Text:
This Technical Specification has been produced by the 3rd Generation Partnership Project (3GPP).
The contents of the present document are subject to continuing work within the TSG and may change following formal TSG approval. Should the TSG modify the contents of the present document, it will be re-released by the TSG with an identifying change of release date and an increase in version number as follows:
Version x.y.z
where:
x the first digit:
1 presented to TSG for information;
2 presented to 

-----------------

In [ ]:
# 3GPP RAG — EMBEDDING + CHROMADB

from pathlib import Path
import json

from sentence_transformers import SentenceTransformer
import chromadb


# 1. PATHS

ROOT_DIR = Path(
    r"C:\Users\ASUS\Desktop\3gpp_specs"
)

INPUT_FILE = ROOT_DIR / "parsed_chunks.json"

CHROMA_DIR = ROOT_DIR / "chroma_db"


# 2. LOAD CHUNKS

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded chunks: {len(chunks)}")


# 3. LOAD EMBEDDING MODEL

#MODEL_NAME = "all-MiniLM-L6-v2"

#embedding_model = SentenceTransformer(MODEL_NAME)

#embedding_model = SentenceTransformer(
#    "sentence-transformers/multi-qa-MiniLM-L6-dot-v1"
#)

embedding_model = SentenceTransformer(
    "sentence-transformers/multi-qa-mpnet-base-dot-v1"
)

print(f"Embedding model: multi-qa-mpnet-base-dot-v1")


# 4. CREATE CHROMADB CLIENT

client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)


# 5. CREATE / GET COLLECTION

collection = client.get_or_create_collection(
    name="3gpp_specs"
)


# 6. PREPARE DATA

ids = []
documents = []
metadatas = []

for chunk in chunks:

    ids.append(chunk["chunk_id"])

    documents.append(chunk["text"])

    metadatas.append({
        "spec": chunk["spec"],
        "source": chunk["source"],
        "section": chunk["section"],
        "title": chunk["title"],
        "parent_section": (
            chunk["parent_section"]
            or ""
        ),
        "parent_title": (
            chunk["parent_title"]
            or ""
        ),
        "chunk_index": chunk["chunk_index"]
    })


# 7. CREATE EMBEDDINGS

print("\nCreating embeddings...")

embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings created.")
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")


# 8. ADD TO CHROMADB

print("\nAdding documents to ChromaDB...")

# ChromaDB can handle batches, which is safer for large datasets.

BATCH_SIZE = 500

for start in range(0, len(chunks), BATCH_SIZE):

    end = start + BATCH_SIZE

    collection.add(
        ids=ids[start:end],
        documents=documents[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end]
    )

    print(
        f"Added {min(end, len(chunks))}/{len(chunks)}"
    )


# 9. CHECK DATABASE

print("\n" + "=" * 70)
print("CHROMADB COMPLETE")
print("=" * 70)

print(f"Collection : {collection.name}")
print(f"Documents  : {collection.count()}")
print(f"Database   : {CHROMA_DIR}")

c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loaded chunks: 15773
Embedding model: multi-qa-mpnet-base-dot-v1

Creating embeddings...


Batches: 100%|██████████| 493/493 [06:56<00:00,  1.18it/s]


Embeddings created.
Number of embeddings: 15773
Embedding dimension: 768

Adding documents to ChromaDB...
Added 500/15773
Added 1000/15773
Added 1500/15773
Added 2000/15773
Added 2500/15773
Added 3000/15773
Added 3500/15773
Added 4000/15773
Added 4500/15773
Added 5000/15773
Added 5500/15773
Added 6000/15773
Added 6500/15773
Added 7000/15773
Added 7500/15773
Added 8000/15773
Added 8500/15773
Added 9000/15773
Added 9500/15773
Added 10000/15773
Added 10500/15773
Added 11000/15773
Added 11500/15773
Added 12000/15773
Added 12500/15773
Added 13000/15773
Added 13500/15773
Added 14000/15773
Added 14500/15773
Added 15000/15773
Added 15500/15773
Added 15773/15773

CHROMADB COMPLETE
Collection : 3gpp_specs
Documents  : 15773
Database   : C:\Users\ASUS\Desktop\3gpp_specs\chroma_db


In [1]:
query = "What is the purpose of the Service Request procedure?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i in range(5):
    print("=" * 80)
    print("Result:", i + 1)
    print("Text:", results["documents"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("Distance:", results["distances"][0][i])

NameError: name 'embedding_model' is not defined

In [ ]:
import os
import re
import sys

from typing import (
    TypedDict,
    List,
    Dict,
    Tuple,
    Optional,
)

from getpass import getpass

import torch

import chromadb

from sentence_transformers import SentenceTransformer

from rank_bm25 import BM25Okapi

from langgraph.graph import (
    StateGraph,
    START,
    END,
)

from langchain_openai import ChatOpenAI

from langchain_huggingface import (
    HuggingFaceEmbeddings,
)

from datasets import Dataset

from ragas import evaluate

from ragas.embeddings import (
    LangchainEmbeddingsWrapper,
)

from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)

from pydantic import (
    BaseModel,
    Field,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import PeftModel


# UTF-8 OUTPUT

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(
        encoding="utf-8"
    )


# 2. CONFIGURATION

# Retrieval configuration

TOP_K = 5

CANDIDATE_K = 40

BM25_CANDIDATE_K = 40

RRF_K = 60


# Agentic retrieval configuration

# Maximum number of retrieval rounds.
#
# Round 1:
#   Original question
#
# Round 2:
#   Original question + Mistral-expanded queries
#
MAX_RETRIEVAL_ATTEMPTS = 2


ROOT_DIR = r"C:\Users\ASUS\Desktop\3gpp_specs"

CHROMA_DIR = os.path.join(
    ROOT_DIR,
    "chroma_db",
)

COLLECTION_NAME = "3gpp_specs"


# Fine-tuned Qwen model

BASE_MODEL = (
    r"C:\Users\ASUS\Desktop"
    r"\telecom_project\models\qwen2.5-1.5b"
)

ADAPTER_DIR = "Tuned_model"


# 4. CITATION PATTERNS

# Expected citation:
#
# [TS 23.288, Section 9.2.3]
#
CITATION_PATTERN = re.compile(
    r"\[\s*(?:TS\s*)?"
    r"(\d{2}\.\d{3})"
    r"[,\s]+"
    r"(?:Section|Clause)\s+"
    r"([\d.]+)"
    r"\s*\]",
    re.IGNORECASE,
)


# Detect:
#
# TS 23.502
#
SPEC_PATTERN = re.compile(
    r"\bTS\s*(\d{2}\.\d{3})\b",
    re.IGNORECASE,
)


# 5. LOAD TOKENIZER

print("=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# 6. 4-BIT QUANTIZATION CONFIG

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)


# 7. LOAD BASE MODEL

print("=" * 70)
print("LOADING BASE QWEN MODEL")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
)


# 8. LOAD QLORA ADAPTER

print("=" * 70)
print("LOADING QLORA ADAPTER")
print("=" * 70)

tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

tuned_model.eval()

print("Fine-tuned model loaded successfully.")


# 9. LOCAL GENERATION HELPER

def ask(
    model,
    question,
    options=None,
    max_new_tokens=256,
):
    """
    Generate text using the supplied HuggingFace model.

    This function is used for:
        1. Router
        2. Final answer generation
    """

    user_content = (
        f"Question:\n{question}"
    )

    if options:

        user_content += (
            "\n\nOptions:\n"
            + "\n".join(
                f"{i + 1}. {o}"
                for i, o in enumerate(options)
            )
        )

    messages = [
        {
            "role": "user",
            "content": user_content,
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # Put inputs on the same device as the model.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


# 10. EMBEDDING MODEL

print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

#embedding_model = SentenceTransformer(
#    "all-MiniLM-L6-v2"
#)
#embedding_model = SentenceTransformer(
#    "sentence-transformers/multi-qa-MiniLM-L6-dot-v1"
#)

embedding_model = SentenceTransformer(
    "sentence-transformers/multi-qa-mpnet-base-dot-v1"
)

# 11. CITATION HELPERS

def _citation(
    item: dict
) -> Optional[str]:

    """
    Build canonical citation from retrieved metadata.
    """

    spec = (
        item.get("spec", "")
        .removeprefix("TS ")
        .strip()
    )

    section = str(
        item.get("section", "")
    ).strip()

    if not spec or not section:
        return None

    return (
        f"[TS {spec}, Section {section}]"
    )


def _validated_citation(
    answer: str,
    context: List[Dict],
) -> Optional[str]:

    """
    Accept only a citation that actually belongs
    to one of the retrieved chunks.
    """

    allowed = {
        _citation(item)
        for item in context
    }

    allowed.discard(None)

    for spec, section in CITATION_PATTERN.findall(
        answer
    ):

        citation = (
            f"[TS {spec}, Section {section}]"
        )

        if citation in allowed:
            return citation

    return None


def _ensure_retrieved_citation(
    answer: str,
    context: List[Dict],
) -> Tuple[str, bool]:

    """
    Keep a verified model citation.

    If the model produced no valid citation,
    attach the highest-ranked retrieved source.

    The citation is selected locally from retrieval
    metadata rather than hallucinated by another model.
    """

    verified = _validated_citation(
        answer,
        context,
    )

    if verified:
        return answer, True

    # Remove unsupported citations.
    cleaned = CITATION_PATTERN.sub(
        "",
        answer,
    ).strip()

    # Do not attach citations to an explicit abstention.
    if (
        "insufficient" in
        cleaned.lower()
    ):
        return cleaned, False

    fallback = next(
        (
            citation
            for item in context
            if (
                citation := _citation(item)
            )
        ),
        None,
    )

    if fallback:

        return (
            f"{cleaned}\n{fallback}",
            False,
        )

    return cleaned, False


# 12. CONNECT TO CHROMADB

print("=" * 70)
print("CONNECTING TO CHROMADB")
print("=" * 70)

client = chromadb.PersistentClient(
    path=CHROMA_DIR
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

print(
    "ChromaDB documents:",
    collection.count(),
)


# ============================================================
# 13. BUILD BM25 INDEX
# ============================================================

print("=" * 70)
print("BUILDING BM25 INDEX")
print("=" * 70)

all_docs = collection.get(
    include=[
        "documents",
        "metadatas",
    ]
)

bm25_documents = all_docs[
    "documents"
]

bm25_metadatas = all_docs[
    "metadatas"
]

bm25_tokenized = [
    re.findall(
        r"\b\w+\b",
        doc.lower(),
    )
    for doc in bm25_documents
]

bm25 = BM25Okapi(
    bm25_tokenized
)

print(
    "BM25 documents:",
    len(bm25_documents),
)


# 14. MISTRAL RETRIEVAL CRITIC

print("=" * 70)
print("CONFIGURING MISTRAL RETRIEVAL CRITIC")
print("=" * 70)

if "MISTRAL_API_KEY" not in os.environ:

    os.environ[
        "MISTRAL_API_KEY"
    ] = getpass(
        "Enter your Mistral API key: "
    )


retrieval_judge = ChatOpenAI(
    model="codestral-2508",
    base_url="https://api.mistral.ai/v1",
    api_key=os.environ[
        "MISTRAL_API_KEY"
    ],
    temperature=0,
)


# 15. STRUCTURED RETRIEVAL EVALUATION

class RetrievalEvaluation(
    BaseModel
):

    relevant: bool = Field(
        description=(
            "True only if the retrieved chunks "
            "contain enough evidence to answer "
            "the question."
        )
    )

    score: float = Field(
        description=(
            "Context sufficiency score from 0 to 1."
        )
    )

    missing_information: str = Field(
        description=(
            "Information needed but missing "
            "from the context."
        )
    )

    search_queries: List[str] = Field(
        description=(
            "Up to three better search queries. "
            "Return [] if the context is sufficient."
        )
    )


retrieval_judge_structured = (
    retrieval_judge.with_structured_output(
        RetrievalEvaluation
    )
)


# 16. MISTRAL RETRIEVAL CRITIC PROMPT

RETRIEVAL_EVALUATION_PROMPT = """
You are the retrieval critic for a 3GPP telecom RAG system.

DO NOT answer the user's question.

Your ONLY job is to determine whether the retrieved context
contains enough evidence to answer the question accurately.

USER QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

Rules:

1. relevant=true ONLY when the retrieved chunks contain enough
   evidence to answer the question reliably.

2. relevant=false when:
   - the chunks are only loosely related,
   - the required procedure is missing,
   - the required message is missing,
   - the required requirement is missing,
   - the context is incomplete,
   - the answer would require guessing,
   - an exact TS/section requested by the user is not represented.

3. Do NOT use your own knowledge to fill missing information.

4. Explain what is missing.

5. If relevant=false, generate up to THREE improved search queries.

6. Use exact 3GPP terminology whenever possible.

7. Preserve an explicitly mentioned TS number.

8. If the context is sufficient, return an empty search_queries list.

9. The score represents CONTEXT SUFFICIENCY,
   not general topic similarity.

Return the requested structured evaluation.
"""


# 17. FORMAT RETRIEVED CONTEXT FOR MISTRAL

def format_context_for_judge(
    context: List[Dict]
) -> str:

    if not context:

        return (
            "NO CONTEXT WAS RETRIEVED."
        )

    return "\n\n".join(
        f"""
[Chunk {i}]

Specification:
{item.get('spec', '')}

Section:
{item.get('section', '')}

Title:
{item.get('title', '')}

Content:
{item.get('text', '')}
"""
        for i, item in enumerate(
            context,
            1,
        )
    )


# 18. EVALUATE RETRIEVAL

def evaluate_retrieval(
    question: str,
    context: List[Dict],
) -> RetrievalEvaluation:

    prompt = (
        RETRIEVAL_EVALUATION_PROMPT.format(
            question=question,
            context=format_context_for_judge(
                context
            ),
        )
    )

    result = (
        retrieval_judge_structured.invoke(
            prompt
        )
    )

    result.score = max(
        0.0,
        min(
            1.0,
            float(result.score),
        ),
    )

    result.search_queries = [
        q.strip()
        for q in result.search_queries
        if q and q.strip()
    ][:3]

    return result


# 19. AGENT STATE

class AgentState(
    TypedDict
):

    question: str

    options: list

    # Router
    use_rag: bool
    route_reason: str

    # Final context
    context: list

    # Final answer
    answer: str

    # Citation status
    cited_by_local_model: bool

    # Retrieval state
    retrieval_attempt: int
    retrieval_query: str
    retrieval_queries: list

    # Mistral retrieval evaluation
    retrieval_relevant: bool
    retrieval_score: float
    missing_information: str

    # Query expansion
    expanded_queries: list

    # Number of corrective retrieval rounds
    retry_count: int


# 20. ROUTER NODE

def route_query(
    state: AgentState
):

    question = state[
        "question"
    ]
    options = state["options"]

    print("\n" + "=" * 70)
    print("AGENT ROUTER")
    print("=" * 70)

    print(
        "Question:",
        question,
    )
    print(
        "Options:",
        options,
    )

    router_prompt = f"""
    return 'NO' if you know the answer to the question: {question} and 'YES' otherwise.

    return ONLY 'YES' or 'NO'.
    Decision:
    """

    decision = ask(
        tuned_model,
        router_prompt,
        max_new_tokens=16,
    ).strip().upper()

    print("*******************************    ",decision,"    *******************************")

 
    use_rag = bool(
    re.search(
        r"\bYES\b",
        decision,
        re.IGNORECASE,
    )
    )
    print("*******************************    ",use_rag,"    *******************************")

    if use_rag:

        reason = (
            "Authoritative 3GPP retrieval is needed."
        )

    else:

        reason = (
            "External 3GPP retrieval is not required."
        )

    print(
        "\nRouter decision:",
        "USE RAG"
        if use_rag
        else
        "NO RAG",
    )

    print(
        "Reason:",
        reason,
    )

    return {

        "use_rag": use_rag,

        "route_reason": reason,

        "retrieval_attempt": 0,

        "retrieval_query": question,

        "retrieval_queries": [],

        "retrieval_relevant": False,

        "retrieval_score": 0.0,

        "missing_information": "",

        "expanded_queries": [],

        "retry_count": 0,
    }


# 21. LOW-LEVEL HYBRID RETRIEVER

def hybrid_retrieve(
    question: str
) -> List[Dict]:

    """
    Hybrid retrieval:

        Dense retrieval
              +
        BM25 retrieval
              |
              v
        Reciprocal Rank Fusion
              |
              v
           TOP_K

    If the question explicitly contains a TS number,
    retrieval is restricted to that specification.
    """

    # DENSE RETRIEVAL

    query_embedding = (
        embedding_model.encode(
            question,
            normalize_embeddings=True,
        ).tolist()
    )

    spec_match = SPEC_PATTERN.search(
        question
    )

    query_args = {

        "query_embeddings": [
            query_embedding
        ],

        "n_results": CANDIDATE_K,
    }

    # Explicit TS filtering

    if spec_match:

        query_args["where"] = {
            "spec": (
                f"TS {spec_match.group(1)}"
            )
        }

    dense_results = collection.query(
        **query_args
    )

    # BM25 RETRIEVAL

    query_tokens = re.findall(
        r"\b\w+\b",
        question.lower(),
    )

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    allowed_indices = list(
        range(
            len(bm25_documents)
        )
    )

    # Explicit TS filtering

    if spec_match:

        target_spec = (
            f"TS {spec_match.group(1)}"
        )

        allowed_indices = [

            i

            for i, metadata
            in enumerate(
                bm25_metadatas
            )

            if metadata.get(
                "spec",
                "",
            )
            == target_spec
        ]

    bm25_ranked_indices = sorted(

        allowed_indices,

        key=lambda i:
        bm25_scores[i],

        reverse=True,

    )[
        :BM25_CANDIDATE_K
    ]

    # MERGE DENSE + BM25

    candidates = {}

    # Dense candidates

    dense_documents = (
        dense_results["documents"][0]
    )

    dense_metadatas = (
        dense_results["metadatas"][0]
    )

    dense_distances = (
        dense_results.get(
            "distances",
            [[]],
        )[0]
    )

    for rank, i in enumerate(
        range(
            len(dense_documents)
        ),
        start=1,
    ):

        metadata = (
            dense_metadatas[i]
        )

        candidate = {

            "text":
                dense_documents[i],

            "spec":
                metadata.get(
                    "spec",
                    "",
                ),

            "section":
                metadata.get(
                    "section",
                    "",
                ),

            "title":
                metadata.get(
                    "title",
                    "",
                ),

            "source":
                metadata.get(
                    "source",
                    "",
                ),

            "distance":
                float(
                    dense_distances[i]
                )
                if i < len(
                    dense_distances
                )
                else None,

            "dense_rank":
                rank,

            "bm25_rank":
                None,

            "bm25_score":
                None,
        }

        key = (

            candidate["spec"],

            candidate["section"],

            candidate["text"],
        )

        candidates[key] = candidate

    # BM25 candidates

    for rank, i in enumerate(
        bm25_ranked_indices,
        start=1,
    ):

        metadata = (
            bm25_metadatas[i]
        )

        key = (

            metadata.get(
                "spec",
                "",
            ),

            metadata.get(
                "section",
                "",
            ),

            bm25_documents[i],
        )

        if key not in candidates:

            candidates[key] = {

                "text":
                    bm25_documents[i],

                "spec":
                    metadata.get(
                        "spec",
                        "",
                    ),

                "section":
                    metadata.get(
                        "section",
                        "",
                    ),

                "title":
                    metadata.get(
                        "title",
                        "",
                    ),

                "source":
                    metadata.get(
                        "source",
                        "",
                    ),

                "distance":
                    None,

                "dense_rank":
                    None,

                "bm25_rank":
                    rank,

                "bm25_score":
                    float(
                        bm25_scores[i]
                    ),
            }

        else:

            candidates[key][
                "bm25_rank"
            ] = rank

            candidates[key][
                "bm25_score"
            ] = float(
                bm25_scores[i]
            )

    # RECIPROCAL RANK FUSION

    for candidate in (
        candidates.values()
    ):

        rrf_score = 0.0

        if (
            candidate.get(
                "dense_rank"
            )
            is not None
        ):

            rrf_score += (
                1.0
                /
                (
                    RRF_K
                    +
                    candidate[
                        "dense_rank"
                    ]
                )
            )

        if (
            candidate.get(
                "bm25_rank"
            )
            is not None
        ):

            rrf_score += (
                1.0
                /
                (
                    RRF_K
                    +
                    candidate[
                        "bm25_rank"
                    ]
                )
            )

        candidate[
            "hybrid_score"
        ] = rrf_score

    # FINAL TOP-K

    return sorted(

        candidates.values(),

        key=lambda x:
        x["hybrid_score"],

        reverse=True,

    )[
        :TOP_K
    ]


# 22. MERGE RESULTS FROM MULTIPLE QUERIES

def merge_retrieval_results(
    result_lists: List[List[Dict]]
) -> List[Dict]:

    """
    Merge retrieval results from:

        Original question
        +
        Mistral-expanded queries

    Each query has its own hybrid ranking.

    A second RRF is then applied across queries.
    """

    merged = {}

    for results in result_lists:

        for rank, item in enumerate(
            results,
            start=1,
        ):

            key = (

                item.get(
                    "spec",
                    "",
                ),

                item.get(
                    "section",
                    "",
                ),

                item.get(
                    "text",
                    "",
                ),
            )

            if key not in merged:

                merged[key] = dict(
                    item
                )

                merged[key][
                    "query_rrf"
                ] = 0.0

                merged[key][
                    "query_hits"
                ] = 0

            merged[key][
                "query_rrf"
            ] += (
                1.0
                /
                (
                    RRF_K
                    +
                    rank
                )
            )

            merged[key][
                "query_hits"
            ] += 1

    # Convert query-level RRF into final score

    for item in merged.values():

        item[
            "hybrid_score"
        ] = item[
            "query_rrf"
        ]

    return sorted(

        merged.values(),

        key=lambda x:
        x["hybrid_score"],

        reverse=True,

    )[
        :TOP_K
    ]


# 23. RETRIEVE NODE

def retrieve(
    state: AgentState
):

    # Safety check

    if not state["use_rag"]:

        return {
            "context": []
        }

    query = state.get(
        "retrieval_query",
        state["question"],
    )

    attempt = state.get(
        "retrieval_attempt",
        0,
    )

    print("\n" + "=" * 70)
    print(
        f"RETRIEVAL ROUND {attempt + 1}"
    )
    print("=" * 70)

    print(
        "Query:",
        query,
    )

    context = hybrid_retrieve(
        query
    )

    queries = list(
        state.get(
            "retrieval_queries",
            [],
        )
    )

    if query not in queries:

        queries.append(
            query
        )

    # Display retrieved documents

    print(
        "\nRetrieved documents:"
    )

    for i, item in enumerate(
        context,
        start=1,
    ):

        print(
            f"\n[{i}] "
            f"{item.get('spec', '')} "
            f"§{item.get('section', '')} "
            f"{item.get('title', '')}"
        )

        print(
            "Dense rank:",
            item.get(
                "dense_rank"
            ),
        )

        print(
            "BM25 rank:",
            item.get(
                "bm25_rank"
            ),
        )

        print(
            "Hybrid RRF:",
            item.get(
                "hybrid_score"
            ),
        )

        distance = item.get(
            "distance"
        )

        if distance is not None:

            print(
                f"Distance: {distance:.4f}"
            )

        else:

            print(
                "Distance: n/a"
            )

        print(
            item.get(
                "text",
                "",
            )[:400]
        )

    return {

        "context":
            context,

        "retrieval_query":
            query,

        "retrieval_queries":
            queries,

        "retrieval_attempt":
            attempt + 1,
    }


# 24. MISTRAL RETRIEVAL EVALUATION NODE

def evaluate_retrieved_context(
    state: AgentState
):

    print("\n" + "=" * 70)
    print("MISTRAL RETRIEVAL CRITIC")
    print("=" * 70)

    evaluation = evaluate_retrieval(

        state["question"],

        state["context"],
    )

    print(
        "Relevant:",
        evaluation.relevant,
    )

    print(
        "Sufficiency score:",
        f"{evaluation.score:.3f}",
    )

    print(
        "Missing information:",
        evaluation.missing_information,
    )

    print(
        "Expanded queries:",
        evaluation.search_queries,
    )

    return {

        "retrieval_relevant":
            evaluation.relevant,

        "retrieval_score":
            evaluation.score,

        "missing_information":
            evaluation.missing_information,

        "expanded_queries":
            evaluation.search_queries,
    }


# 25. CONDITIONAL DECISION AFTER RETRIEVAL CRITIC

def after_retrieval_evaluation(
    state: AgentState
):

    # Retrieval is sufficient

    if state[
        "retrieval_relevant"
    ]:

        print(
            "\nMistral accepted the retrieved context."
        )

        return "generate"

    # Maximum retrieval attempts reached

    if (
        state["retrieval_attempt"]
        >= MAX_RETRIEVAL_ATTEMPTS
    ):

        print(
            "\nMaximum retrieval attempts reached."
        )

        return "generate"

    # Mistral did not provide useful queries

    if not state[
        "expanded_queries"
    ]:

        print(
            "\nMistral did not produce another query."
        )

        return "generate"

    # Retry

    print(
        "\nRetrieved context judged insufficient."
    )

    print(
        "Starting corrective retrieval..."
    )

    return "prepare_retry"


# 26. PREPARE RETRY

def prepare_retry(
    state: AgentState
):

    queries = state[
        "expanded_queries"
    ]

    print("\n" + "=" * 70)
    print("QUERY EXPANSION")
    print("=" * 70)

    for i, query in enumerate(
        queries,
        start=1,
    ):

        print(
            f"{i}. {query}"
        )

    # We store the first query for bookkeeping.
    #
    # The actual retry node below uses ALL expanded
    # queries together with the original question.

    return {

        "retrieval_query":
            queries[0],

        "retry_count":
            state["retry_count"] + 1,
    }


# 27. CORRECTIVE MULTI-QUERY RETRIEVAL

def retrieve_retry_queries(
    state: AgentState
):

    """
    Corrective retrieval performs:

        original question
             +
        expanded query 1
             +
        expanded query 2
             +
        expanded query 3

    Each query gets:
        Dense + BM25 + RRF

    Then all query results are merged using
    another query-level RRF.
    """

    queries = [

        state["question"]

    ]

    queries.extend(
        state.get(
            "expanded_queries",
            [],
        )
    )

    # Remove duplicates

    queries = list(
        dict.fromkeys(

            q.strip()

            for q in queries

            if q and q.strip()
        )
    )

    print("\n" + "=" * 70)
    print("CORRECTIVE MULTI-QUERY RETRIEVAL")
    print("=" * 70)

    print(
        "Queries:"
    )

    for i, query in enumerate(
        queries,
        start=1,
    ):

        print(
            f"{i}. {query}"
        )

    result_lists = []

    # Retrieve every query

    for query in queries:

        print(
            "\nRetrieving:",
            query,
        )

        results = hybrid_retrieve(
            query
        )

        result_lists.append(
            results
        )

    # Merge using query-level RRF

    context = merge_retrieval_results(
        result_lists
    )

    # Update query history

    all_queries = list(
        state.get(
            "retrieval_queries",
            [],
        )
    )

    for query in queries:

        if query not in all_queries:

            all_queries.append(
                query
            )

    # Display final corrective context

    print(
        "\nFinal corrective context:"
    )

    for i, item in enumerate(
        context,
        start=1,
    ):

        print(
            f"\n[{i}] "
            f"{item.get('spec', '')} "
            f"§{item.get('section', '')} "
            f"{item.get('title', '')}"
        )

        print(
            "Query-RRF:",
            item.get(
                "hybrid_score"
            ),
        )

        print(
            "Query hits:",
            item.get(
                "query_hits"
            ),
        )

        print(
            item.get(
                "text",
                "",
            )[:400]
        )

    return {

        "context":
            context,

        "retrieval_query":
            queries[0],

        "retrieval_queries":
            all_queries,

        "retrieval_attempt":
            state[
                "retrieval_attempt"
            ] + 1,
    }


# 28. GENERATE ANSWER NODE

def generate(
    state: AgentState
):

    question = state[
        "question"
    ]

    options = state["options"]

    context = state[
        "context"
    ]

    print("\n" + "=" * 70)
    print("GENERATING ANSWER")
    print("=" * 70)

    # CASE 1:
    # NO RAG

    if not state[
        "use_rag"
    ]:

        prompt = f"""
You are a telecom and AI assistant.

Answer the user's question directly using your learned knowledge.

No 3GPP retrieval was performed.

Do not pretend that you retrieved information from the
3GPP knowledge base.

Do not invent 3GPP citations.

If the user requires an exact 3GPP clause or standards-specific
answer, say that the 3GPP knowledge base should be consulted
rather than inventing a clause.

QUESTION:

{question}

OPTIONS:
{options}

ANSWER:
"""

        answer = ask(
            tuned_model,
            prompt,
            max_new_tokens=256,
        )

        print(
            "RAG used: NO"
        )

        print(
            "Answer:",
            answer,
        )

        return {

            "answer":
                answer,

            "cited_by_local_model":
                False,
        }

    # CASE 2:
    # RAG

    # Format context

    if context:

        context_text = "\n\n".join(

            f"""
3GPP SPECIFICATION:
{item['spec']}

SECTION:
{item['section']} — {item['title']}

CONTENT:
{item['text']}
"""

            for item in context
        )

    else:

        context_text = (
            "NO 3GPP CONTEXT WAS RETRIEVED."
        )

    # Retrieval status

    if state[
        "retrieval_relevant"
    ]:

        evidence_instruction = (
            "The retrieval critic judged the context sufficient."
        )

    else:

        evidence_instruction = """
The retrieval critic could NOT confirm that the context is
sufficient after the allowed retrieval attempts.

Do NOT fill missing information from model memory.

If the evidence is insufficient, explicitly say:

"The retrieved 3GPP context is insufficient to answer reliably."
"""

    # --------------------------------------------------------
    # Generation prompt
    # --------------------------------------------------------

    prompt = f"""
You are a telecom standards assistant.

{evidence_instruction}

Answer the question using ONLY the retrieved 3GPP context.

QUESTION:

{question}

OPTIONS:
{options}

3GPP CONTEXT:

{context_text}

Rules:

1. Do not use unsupported knowledge.

2. Do not guess.

3. If the context is insufficient, say exactly:

"The retrieved 3GPP context is insufficient to answer reliably."

4. Answer in one or two complete sentences.

5. Do not answer with an option letter.

6. Do not answer in multiple-choice style.

7. If the context supports the answer, end with a citation
   in exactly this format:

[TS XX.XXX, Section X.X.X]

8. Do not invent a citation.

9. The citation must correspond to a retrieved chunk.

ANSWER:
"""

    answer = ask(
        tuned_model,
        prompt,
        max_new_tokens=256,
    )

    # Validate citation

    answer, cited = (
        _ensure_retrieved_citation(
            answer,
            context,
        )
    )

    print(
        "RAG used: YES"
    )

    print(
        "Retrieval attempts:",
        state[
            "retrieval_attempt"
        ],
    )

    print(
        "Retrieval accepted:",
        state[
            "retrieval_relevant"
        ],
    )

    print(
        "Retrieval score:",
        state[
            "retrieval_score"
        ],
    )

    print(
        "Citation supplied by local model:",
        cited,
    )

    print(
        "Final answer:",
        answer,
    )

    return {

        "answer":
            answer,

        "cited_by_local_model":
            cited,
    }


# 29. BUILD LANGGRAPH

print("=" * 70)
print("BUILDING LANGGRAPH")
print("=" * 70)

graph = StateGraph(
    AgentState
)


# Nodes

graph.add_node(
    "route",
    route_query,
)

graph.add_node(
    "retrieve",
    retrieve,
)

graph.add_node(
    "evaluate_retrieval",
    evaluate_retrieved_context,
)

graph.add_node(
    "prepare_retry",
    prepare_retry,
)

graph.add_node(
    "retrieve_retry",
    retrieve_retry_queries,
)

graph.add_node(
    "generate",
    generate,
)


# START -> ROUTER

graph.add_edge(
    START,
    "route",
)


# ROUTER CONDITIONAL EDGE

def after_route(
    state: AgentState
):

    if state[
        "use_rag"
    ]:

        return "retrieve"

    return "generate"


graph.add_conditional_edges(

    "route",

    after_route,

    {

        "retrieve":
            "retrieve",

        "generate":
            "generate",
    },
)


# FIRST RETRIEVAL

graph.add_edge(
    "retrieve",
    "evaluate_retrieval",
)


# RETRIEVAL CRITIC

graph.add_conditional_edges(

    "evaluate_retrieval",

    after_retrieval_evaluation,

    {

        "generate":
            "generate",

        "prepare_retry":
            "prepare_retry",
    },
)


# CORRECTIVE RETRIEVAL

graph.add_edge(
    "prepare_retry",
    "retrieve_retry",
)

graph.add_edge(
    "retrieve_retry",
    "evaluate_retrieval",
)


# GENERATION -> END

graph.add_edge(
    "generate",
    END,
)


# COMPILE

app = graph.compile()

print(
    "LangGraph compiled successfully."
)


# 30. TEST SET

TEST_QUESTIONS = [

    {
        "question":
            "Which network function provides Network Data Analytics services in the 5G System?",
        "options": [
            "NSSF",
            "NWDAF",
            "NEF",
            "PCF",
        ],
        "ground_truth":
            "NWDAF",
    },

    {
        "question":
            "Which interface is used by NWDAF to request subscription to data delivery from a data source?",
        "options": [
            "Nnrf",
            "Nnwdaf",
            "Nnf",
            "Nnssf",
        ],
        "ground_truth":
            "Nnf interface",
    },

    {
        "question":
            "What type of information can NWDAF provide to a consumer through an analytics subscription?",
        "options": [
            "Network analytics information",
            "UE authentication credentials",
            "User-plane packets",
            "Radio bearer configuration",
        ],
        "ground_truth":
            "Network analytics information",
    },

    {
        "question":
            "Which 5G function can consume analytics information provided by NWDAF to support network optimization?",
        "options": [
            "Only the UPF",
            "Only the AMF",
            "Network functions such as AMF, SMF, PCF, etc., depending on the analytics service",
            "Only the gNB",
        ],
        "ground_truth":
            "Network functions such as AMF, SMF, PCF, etc., depending on the analytics service",
    },

    {
        "question":
            "What is the purpose of the NWDAF in the 5G System architecture?",
        "options": [
            "To provide network data analytics services based on data collected from 5GS data sources",
            "To provide radio access to UEs",
            "To manage user-plane packet forwarding",
            "To perform subscriber authentication only",
        ],
        "ground_truth":
            "To provide network data analytics services based on data collected from 5GS data sources",
    },

    {
        "question":
            "What is the role of the AMF in the 5G System?",
        "options": [
            "User Plane Function",
            "Access and Mobility Management Function",
            "Policy Control Function",
            "Network Exposure Function",
        ],
        "ground_truth":
            "Access and Mobility Management Function",
    },

    {
        "question":
            "Which network function is responsible for establishing and managing PDU sessions?",
        "options": [
            "AMF",
            "UPF",
            "SMF",
            "UDM",
        ],
        "ground_truth":
            "SMF",
    },

    {
        "question":
            "Which interface connects the UE and AMF for NAS signalling?",
        "options": [
            "N2",
            "N1",
            "N3",
            "N4",
        ],
        "ground_truth":
            "N1",
    },

    {
        "question":
            "Which interface is used between the (R)AN and AMF?",
        "options": [
            "N3",
            "N1",
            "N2",
            "N6",
        ],
        "ground_truth":
            "N2",
    },

    {
        "question":
            "Which interface is used between the (R)AN and UPF for user-plane traffic?",
        "options": [
            "N3",
            "N2",
            "N4",
            "N6",
        ],
        "ground_truth":
            "N3",
    },

    {
        "question":
            "What is the primary purpose of the UPF in the 5GS architecture?",
        "options": [
            "Subscriber authentication",
            "User-plane packet handling and forwarding",
            "Access and mobility management",
            "Policy control",
        ],
        "ground_truth":
            "User-plane packet handling and forwarding",
    },

    {
        "question":
            "Which function is responsible for policy control in the 5G Core?",
        "options": [
            "UDM",
            "PCF",
            "AUSF",
            "SMF",
        ],
        "ground_truth":
            "PCF",
    },

    {
        "question":
            "What is the role of the UDM in the 5G Core?",
        "options": [
            "Unified Data Management",
            "User Plane Management",
            "Unified Policy Control",
            "User Equipment Mobility",
        ],
        "ground_truth":
            "Unified Data Management",
    },

    {
        "question":
            "Which procedure is used by a UE to register with the 5G System?",
        "options": [
            "Service Request procedure",
            "Deregistration procedure",
            "Registration procedure",
            "PDU Session Release procedure",
        ],
        "ground_truth":
            "Registration procedure",
    },

    {
        "question":
            "What procedure is used to establish a PDU Session between the UE and the 5G Core?",
        "options": [
            "Registration procedure",
            "PDU Session Establishment procedure",
            "Service Request procedure",
            "Deregistration procedure",
        ],
        "ground_truth":
            "PDU Session Establishment procedure",
    },

    {
        "question":
            "Which network function typically handles the PDU Session Establishment procedure?",
        "options": [
            "AMF",
            "PCF",
            "SMF",
            "AUSF",
        ],
        "ground_truth":
            "SMF",
    },

    {
        "question":
            "What message does the UE send to initiate the 5GS registration procedure?",
        "options": [
            "Registration Accept",
            "Service Request",
            "Registration Request",
            "Deregistration Request",
        ],
        "ground_truth":
            "Registration Request",
    },

    {
        "question":
            "What is the purpose of the Service Request procedure in 5GS?",
        "options": [
            "To request establishment or resumption of signalling or user-plane connectivity for a UE in relevant states",
            "To permanently deregister the UE",
            "To authenticate the UDM",
            "To establish an Xn connection between gNBs",
        ],
        "ground_truth":
            "To request establishment or resumption of signalling or user-plane connectivity for a UE in relevant states",
    },

    {
        "question":
            "What procedure is used when a UE releases its registration from the 5G System?",
        "options": [
            "Registration procedure",
            "Deregistration procedure",
            "Service Request procedure",
            "PDU Session Establishment procedure",
        ],
        "ground_truth":
            "Deregistration procedure",
    },

    {
        "question":
            "During PDU Session Establishment, which function selects and controls the UPF?",
        "options": [
            "AMF",
            "SMF",
            "PCF",
            "UDM",
        ],
        "ground_truth":
            "SMF",
    },

    {
        "question":
            "What protocol is specified by 3GPP TS 24.501?",
        "options": [
            "NR RRC protocol",
            "NGAP protocol",
            "Non-Access-Stratum (NAS) protocol for 5GS",
            "GTP-U protocol",
        ],
        "ground_truth":
            "Non-Access-Stratum (NAS) protocol for 5GS",
    },

    {
        "question":
            "Which NAS message is used by the UE to request registration to the 5G System?",
        "options": [
            "Registration Accept",
            "Registration Request",
            "Service Accept",
            "PDU Session Establishment Request",
        ],
        "ground_truth":
            "Registration Request",
    },

    {
        "question":
            "Which NAS message is used by the network to accept a successful registration?",
        "options": [
            "Registration Request",
            "Service Accept",
            "Registration Accept",
            "Deregistration Accept",
        ],
        "ground_truth":
            "Registration Accept",
    },

    {
        "question":
            "Which NAS procedure allows a UE to request services while it is already registered?",
        "options": [
            "Registration procedure",
            "Deregistration procedure",
            "Service Request procedure",
            "Authentication procedure",
        ],
        "ground_truth":
            "Service Request procedure",
    },

    {
        "question":
            "Which NAS message is associated with establishing a PDU Session?",
        "options": [
            "PDU Session Establishment Request",
            "Registration Request",
            "Service Request",
            "PDU Session Release Command",
        ],
        "ground_truth":
            "PDU Session Establishment Request",
    },

    {
        "question":
            "Which NAS protocol layer handles 5GS mobility-management signalling?",
        "options": [
            "5GSM",
            "RRC",
            "5GMM",
            "NGAP",
        ],
        "ground_truth":
            "5GMM",
    },

    {
        "question":
            "Which NAS protocol layer handles 5GS session-management signalling?",
        "options": [
            "5GMM",
            "5GSM",
            "RRC",
            "NAS-SEC",
        ],
        "ground_truth":
            "5GSM",
    },

    {
        "question":
            "What is the purpose of the SUCI in 5G security?",
        "options": [
            "To identify the serving AMF",
            "To conceal the permanent subscription identifier (SUPI)",
            "To identify the UPF",
            "To encrypt user-plane traffic",
        ],
        "ground_truth":
            "To conceal the permanent subscription identifier (SUPI)",
    },

    {
        "question":
            "Which identifier is protected when the UE uses SUCI instead of directly exposing the SUPI?",
        "options": [
            "GUTI",
            "PEI",
            "SUPI",
            "AMF ID",
        ],
        "ground_truth":
            "SUPI",
    },

    {
        "question":
            "Which authentication methods are specified for 5G System authentication?",
        "options": [
            "5G-AKA and EAP-based authentication methods, including EAP-AKA'",
            "Only TLS authentication",
            "Only IPsec authentication",
            "Only OAuth authentication",
        ],
        "ground_truth":
            "5G-AKA and EAP-based authentication methods, including EAP-AKA'",
    },

    {
        "question":
            "What is the role of the SEAF during 5G authentication?",
        "options": [
            "It provides user-plane packet forwarding",
            "It serves as the security anchor function in the serving network during authentication",
            "It manages PDU Session resources",
            "It performs radio resource scheduling",
        ],
        "ground_truth":
            "It serves as the security anchor function in the serving network during authentication",
    },

    {
        "question":
            "Which network function is responsible for authentication-related functions in the 5G Core?",
        "options": [
            "AUSF",
            "AMF",
            "SMF",
            "PCF",
        ],
        "ground_truth":
            "AUSF",
    },

    {
        "question":
            "What is the purpose of NAS security in 5GS?",
        "options": [
            "To provide protection such as integrity and confidentiality for NAS signalling",
            "To forward user-plane packets",
            "To select the UPF",
            "To perform radio scheduling",
        ],
        "ground_truth":
            "To provide protection such as integrity and confidentiality for NAS signalling",
    },

    {
        "question":
            "What does 3GPP TS 38.300 specify?",
        "options": [
            "5G Core network procedures",
            "NR and NG-RAN overall description and Stage-2 aspects",
            "5GS NAS protocol",
            "5G security architecture",
        ],
        "ground_truth":
            "NR and NG-RAN overall description and Stage-2 aspects",
    },

    {
        "question":
            "What are the main types of NG-RAN nodes specified for 5GS?",
        "options": [
            "gNB and ng-eNB",
            "AMF and SMF",
            "UPF and UDM",
            "NRF and PCF",
        ],
        "ground_truth":
            "gNB and ng-eNB",
    },

    {
        "question":
            "Which NG-RAN node provides NR radio access to the UE?",
        "options": [
            "ng-eNB",
            "AMF",
            "gNB",
            "UPF",
        ],
        "ground_truth":
            "gNB",
    },

    {
        "question":
            "What is the purpose of the NG interface in NG-RAN?",
        "options": [
            "To connect UEs directly to the UPF",
            "To connect NG-RAN nodes to the 5G Core",
            "To connect gNBs to each other",
            "To connect the UE to the AMF for NAS",
        ],
        "ground_truth":
            "To connect NG-RAN nodes to the 5G Core",
    },

    {
        "question":
            "Which interface connects a gNB to another gNB for inter-node communication?",
        "options": [
            "N2",
            "N3",
            "Xn",
            "N1",
        ],
        "ground_truth":
            "Xn",
    },

    {
        "question":
            "What are the two broad planes used to describe communication between network entities?",
        "options": [
            "Radio plane and core plane",
            "Control plane and user plane",
            "Access plane and mobility plane",
            "Session plane and security plane",
        ],
        "ground_truth":
            "Control plane and user plane",
    },

    {
        "question":
            "What protocol is specified by 3GPP TS 38.331?",
        "options": [
            "NGAP",
            "NAS",
            "NR Radio Resource Control (RRC)",
            "GTP-U",
        ],
        "ground_truth":
            "NR Radio Resource Control (RRC)",
    },

    {
        "question":
            "What are the main RRC states defined for NR?",
        "options": [
            "RRC_IDLE, RRC_INACTIVE, and RRC_CONNECTED",
            "RRC_IDLE and RRC_ACTIVE only",
            "RRC_CONNECTED and RRC_SLEEP only",
            "RRC_INIT, RRC_ACTIVE, and RRC_RELEASED",
        ],
        "ground_truth":
            "RRC_IDLE, RRC_INACTIVE, and RRC_CONNECTED",
    },

    {
        "question":
            "Which RRC message is used to establish an RRC connection?",
        "options": [
            "RRCSetupRequest",
            "RRCReconfiguration",
            "RRCSetup",
            "RRCRelease",
        ],
        "ground_truth":
            "RRCSetup",
    },

    {
        "question":
            "Which RRC message is used by the UE to request establishment of an RRC connection?",
        "options": [
            "RRCSetup",
            "RRCSetupRequest",
            "RRCReestablishment",
            "RRCResume",
        ],
        "ground_truth":
            "RRCSetupRequest",
    },

    {
        "question":
            "Which RRC message is used by the UE to report measurement results to the network?",
        "options": [
            "MeasurementReport",
            "RRCSetupRequest",
            "RRCRelease",
            "RRCResumeRequest",
        ],
        "ground_truth":
            "MeasurementReport",
    },

    {
        "question":
            "Which RRC message is commonly used by the network to configure or reconfigure radio resources?",
        "options": [
            "RRCSetupRequest",
            "MeasurementReport",
            "RRCReconfiguration",
            "RRCRelease",
        ],
        "ground_truth":
            "RRCReconfiguration",
    },

    {
        "question":
            "What is the purpose of RRC measurement reporting?",
        "options": [
            "To authenticate the UE",
            "To provide the network with radio measurements used for mobility and radio-resource decisions",
            "To establish the PDU Session",
            "To forward user-plane packets",
        ],
        "ground_truth":
            "To provide the network with radio measurements used for mobility and radio-resource decisions",
    },
]



# ANSWER ACCURACY HELPER

def is_answer_correct(answer, ground_truth, options):
    """
    Check whether the model's answer matches the ground-truth
    option.

    Uses normalized text matching.
    """

    answer_norm = answer.strip().lower()
    truth_norm = ground_truth.strip().lower()

    # Direct match
    if truth_norm in answer_norm:
        return True

    # If the model gives an option number, check it
    for i, option in enumerate(options, start=1):

        option_norm = option.strip().lower()

        if option_norm == truth_norm:

            # Match forms such as:
            # "1"
            # "Option 1"
            # "1."
            # "The answer is 1"
            patterns = [
                rf"^\s*{i}\s*$",
                rf"^\s*option\s+{i}\s*$",
                rf"^\s*{i}\.",
                rf"\boption\s+{i}\b",
            ]

            for pattern in patterns:
                if re.search(
                    pattern,
                    answer_norm,
                    re.IGNORECASE
                ):
                    return True

    return False


# 31. RUN THE AGENT

agent_results = []


for question_index, test_case in enumerate(
    TEST_QUESTIONS,
    start=1,
):

    print("\n\n")
    print("#" * 70)
    print(
        f"QUESTION {question_index}/{len(TEST_QUESTIONS)}"
    )
    print("#" * 70)

    print(
        test_case["question"]
    )


    # Initial state

    initial_state = {

        "question":
            test_case["question"],

        "options":
            test_case["options"],

        "use_rag":
            False,

        "route_reason":
            "",

        "context":
            [],

        "answer":
            "",

        "cited_by_local_model":
            False,

        "retrieval_attempt":
            0,

        "retrieval_query":
            test_case["question"],

        "retrieval_queries":
            [],

        "retrieval_relevant":
            False,

        "retrieval_score":
            0.0,

        "missing_information":
            "",

        "expanded_queries":
            [],

        "retry_count":
            0,
    }

    # Invoke LangGraph

    result = app.invoke(
        initial_state
    )

    # Save results

    agent_results.append({

        "question":
            test_case["question"],

        "answer":
            result[
                "answer"
            ],

        "contexts": [

            item["text"]

            for item
            in result["context"]
        ],

        "sources": [

            f"{item['spec']} "
            f"§{item['section']}"

            for item
            in result["context"]
        ],

        "ground_truth":
            test_case[
                "ground_truth"
            ],

        "use_rag":
            result[
                "use_rag"
            ],
        "correct":
            is_answer_correct(
            result["answer"],
            test_case["ground_truth"],
            test_case["options"]
        ),

        "route_reason":
            result[
                "route_reason"
            ],

        "cited_by_local_model":
            result[
                "cited_by_local_model"
            ],

        "retrieved_specs": [

            f"{item['spec']} "
            f"§{item['section']}"

            for item
            in result["context"]
        ],

        "retrieval_attempts":
            result.get(
                "retrieval_attempt",
                0,
            ),

        "retrieval_queries":
            result.get(
                "retrieval_queries",
                [],
            ),

        "retrieval_relevant":
            result.get(
                "retrieval_relevant",
                False,
            ),

        "retrieval_score":
            result.get(
                "retrieval_score",
                0.0,
            ),

        "missing_information":
            result.get(
                "missing_information",
                "",
            ),

        "retry_count":
            result.get(
                "retry_count",
                0,
            ),
    })


# 32. FINAL ANSWERS + CITATIONS

print("\n\n")
print("=" * 70)
print("FINAL ANSWERS")
print("=" * 70)


for index, result in enumerate(
    agent_results,
    start=1,
):

    print("\n")
    print("-" * 70)
    print(
        f"QUESTION {index}"
    )
    print("-" * 70)

    print(
        result["question"]
    )

    print("\nANSWER:")
    print(
        result["answer"]
    )

    print("\nRAG USED:")
    print(
        result["use_rag"]
    )

    print("\nROUTE REASON:")
    print(
        result["route_reason"]
    )

    print("\nRETRIEVAL ATTEMPTS:")
    print(
        result[
            "retrieval_attempts"
        ]
    )

    print("\nRETRIEVAL QUERIES:")

    for query in result[
        "retrieval_queries"
    ]:

        print(
            " -",
            query,
        )

    print(
        "\nRETRIEVAL SCORE:",
        result[
            "retrieval_score"
        ],
    )

    print(
        "\nRETRIEVAL ACCEPTED:",
        result[
            "retrieval_relevant"
        ],
    )

    print(
        "\nCITATION SUPPLIED BY LOCAL MODEL:",
        result[
            "cited_by_local_model"
        ],
    )

    print(
        "\nSPEC SECTIONS RETRIEVED:"
    )

    for source in result[
        "retrieved_specs"
    ]:

        print(
            " -",
            source,
        )

    citations = (
        CITATION_PATTERN.findall(
            result["answer"]
        )
    )

    print(
        "\nCITATIONS FOUND:"
    )

    print(
        citations
        if citations
        else
        "NONE"
    )



# 33. ACCURACY BY ROUTING TYPE

rag_results = [
    r
    for r in agent_results
    if r["use_rag"]
]

no_rag_results = [
    r
    for r in agent_results
    if not r["use_rag"]
]


# RAG accuracy

rag_correct = sum(
    r["correct"]
    for r in rag_results
)

rag_total = len(rag_results)

rag_accuracy = (
    rag_correct / rag_total
    if rag_total > 0
    else 0.0
)


# NO-RAG accuracy

no_rag_correct = sum(
    r["correct"]
    for r in no_rag_results
)

no_rag_total = len(no_rag_results)

no_rag_accuracy = (
    no_rag_correct / no_rag_total
    if no_rag_total > 0
    else 0.0
)


# Overall accuracy

total_correct = sum(
    r["correct"]
    for r in agent_results
)

total_questions = len(agent_results)

overall_accuracy = (
    total_correct / total_questions
    if total_questions > 0
    else 0.0
)



print("\n")
print("=" * 70)
print("ANSWER ACCURACY BY RAG ROUTING")
print("=" * 70)

print(
    f"Total questions : {total_questions}"
)

print(
    f"Total correct   : {total_correct}"
)

print(
    f"Overall accuracy: {overall_accuracy:.2%}"
)

print("\n" + "-" * 70)

print(
    f"RAG questions   : {rag_total}"
)

print(
    f"RAG correct     : {rag_correct}"
)

print(
    f"RAG accuracy    : {rag_accuracy:.2%}"
)

print("\n" + "-" * 70)

print(
    f"No-RAG questions: {no_rag_total}"
)

print(
    f"No-RAG correct  : {no_rag_correct}"
)

print(
    f"No-RAG accuracy : {no_rag_accuracy:.2%}"
)

print("=" * 70)


# 33. RAG-ONLY DATASET

rag_results = [

    result

    for result
    in agent_results

    if result[
        "use_rag"
    ]
]


print("\n")
print("=" * 70)
print("RAGAS EVALUATION SPLIT")
print("=" * 70)

print(
    "Total questions:",
    len(agent_results),
)

print(
    "Questions routed to RAG:",
    len(rag_results),
)

print(
    "Questions routed without RAG:",
    len(agent_results)
    -
    len(rag_results),
)


# 34. RAGAS EVALUATION

if rag_results:

    print("\n")
    print("=" * 70)
    print("PREPARING RAGAS")
    print("=" * 70)

    ragas_data = {

        "question": [

            r["question"]

            for r in rag_results
        ],

        "contexts": [

            r["contexts"]

            for r in rag_results
        ],

        "answer": [

            r["answer"]

            for r in rag_results
        ],

        "ground_truth": [

            r["ground_truth"]

            for r in rag_results
        ],
    }

    dataset_rag = Dataset.from_dict(
        ragas_data
    )

    # RAGAS embedding model

    ragas_embeddings = (
        HuggingFaceEmbeddings(
            model_name=(
                "sentence-transformers/"
                "multi-qa-mpnet-base-dot-v1"
            )
        )
    )

    ragas_embeddings = (
        LangchainEmbeddingsWrapper(
            ragas_embeddings
        )
    )

    print(
        "Running RAGAS..."
    )

    # RAGAS
    

    ragas_result = evaluate(

        dataset_rag,

        metrics=[

            context_precision,

            context_recall,

            faithfulness,

            answer_relevancy,
        ],

        llm=retrieval_judge,

        embeddings=ragas_embeddings,
    )

    print("\n")
    print("=" * 70)
    print("RAG-ONLY RAGAS RESULTS")
    print("=" * 70)

    print(
        ragas_result
    )

else:

    print("\n")
    print("=" * 70)
    print("RAGAS SKIPPED")
    print("=" * 70)

    print(
        "No questions were routed to RAG."
    )


# 35. AGENTIC RAG SUMMARY

if rag_results:

    # Initial retrieval accepted immediately

    initial_accept = sum(

        1

        for r
        in rag_results

        if (
            r[
                "retrieval_attempts"
            ]
            == 1
            and
            r[
                "retrieval_relevant"
            ]
        )
    )

    # Questions requiring corrective retrieval

    retries = sum(

        1

        for r
        in rag_results

        if r[
            "retry_count"
        ] > 0
    )

    # Final accepted

    final_accepted = sum(

        1

        for r
        in rag_results

        if r[
            "retrieval_relevant"
        ]
    )

    # Still insufficient

    final_insufficient = sum(

        1

        for r
        in rag_results

        if not r[
            "retrieval_relevant"
        ]
    )

    # Average retrieval score

    average_retrieval_score = (

        sum(
            r[
                "retrieval_score"
            ]

            for r
            in rag_results
        )

        /

        len(
            rag_results
        )
    )

    # Citation statistics

    citations_from_model = sum(

        1

        for r
        in rag_results

        if r[
            "cited_by_local_model"
        ]
    )

    # Print summary

    print("\n")
    print("=" * 70)
    print("AGENTIC RAG SUMMARY")
    print("=" * 70)

    print(
        "Total questions:",
        len(agent_results),
    )

    print(
        "RAG questions:",
        len(rag_results),
    )

    print(
        "No-RAG questions:",
        len(agent_results)
        -
        len(rag_results),
    )

    print(
        "\nAccepted after initial retrieval:",
        initial_accept,

        f"({initial_accept / len(rag_results):.1%})",
    )

    print(
        "\nRequired corrective retrieval:",
        retries,

        f"({retries / len(rag_results):.1%})",
    )

    print(
        "\nAccepted after correction:",
        final_accepted,

        f"({final_accepted / len(rag_results):.1%})",
    )

    print(
        "\nStill insufficient:",
        final_insufficient,

        f"({final_insufficient / len(rag_results):.1%})",
    )

    print(
        "\nAverage Mistral retrieval sufficiency:",
        f"{average_retrieval_score:.3f}",
    )

    print(
        "\nAnswers where local Qwen supplied a verified citation:",
        citations_from_model,

        f"({citations_from_model / len(rag_results):.1%})",
    )


# 36. SIMPLE ROUTING SUMMARY

rag_count = sum(
    1
    for r in agent_results
    if r["use_rag"]
)

no_rag_count = (
    len(agent_results)
    -
    rag_count
)

print("\n")
print("=" * 70)
print("ROUTING SUMMARY")
print("=" * 70)

print(
    f"Total questions: {len(agent_results)}"
)

print(
    f"RAG selected: {rag_count}"
)

print(
    f"RAG skipped: {no_rag_count}"
)



print("\n")
print("=" * 70)
print("TELECOM AGENT EVALUATION COMPLETE")
print("=" * 70)

c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



LOADING TOKENIZER


`torch_dtype` is deprecated! Use `dtype` instead!


LOADING BASE QWEN MODEL
LOADING QLORA ADAPTER


c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\peft\config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'kasa_config', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Fine-tuned model loaded successfully.
LOADING EMBEDDING MODEL
CONNECTING TO CHROMADB
ChromaDB documents: 15773
BUILDING BM25 INDEX
BM25 documents: 15773
CONFIGURING MISTRAL RETRIEVAL CRITIC


... (90 outputs left out)

c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=RetrievalEvaluation(relev...n 3GPP specifications']), input_type=RetrievalEvaluation])
  return self.__pydantic_serializer__.to_python(


Relevant: False
Sufficiency score: 0.000
Missing information: The purpose of RRC measurement reporting is not explicitly stated in the retrieved context. The context describes the process of measurement reporting but does not provide the specific purpose of this procedure.
Expanded queries: ['What is the purpose of RRC measurement reporting in 3GPP TS 38.331', 'What is the purpose of measurement reporting in 3GPP TS 38.300', 'What is the purpose of RRC measurement reporting in 3GPP specifications']

Maximum retrieval attempts reached.

GENERATING ANSWER
RAG used: YES
Retrieval attempts: 2
Retrieval accepted: False
Retrieval score: 0.0
Citation supplied by local model: False
Final answer: option 2: To provide the network with radio measurements used for mobility and radio-resource decisions
[TS 38.300, Section 9.2.4]



FINAL ANSWERS


----------------------------------------------------------------------
QUESTION 1
----------------------------------------------------------------------


Evaluating: 100%|██████████| 92/92 [01:29<00:00,  1.03it/s]




RAG-ONLY RAGAS RESULTS
{'context_precision': 0.6836, 'context_recall': 0.7391, 'faithfulness': 0.2899, 'answer_relevancy': 0.5086}


AGENTIC RAG SUMMARY
Total questions: 46
RAG questions: 23
No-RAG questions: 23

Accepted after initial retrieval: 1 (4.3%)

Required corrective retrieval: 22 (95.7%)

Accepted after correction: 4 (17.4%)

Still insufficient: 19 (82.6%)

Average Mistral retrieval sufficiency: 0.183

Answers where local Qwen supplied a verified citation: 0 (0.0%)


ROUTING SUMMARY
Total questions: 46
RAG selected: 23
RAG skipped: 23


TELECOM AGENT EVALUATION COMPLETE


In [ ]:
# ============================================================
# EXPERIMENT A
# BASE MODEL vs FINE-TUNED MODEL
#
# Qwen2.5-1.5B-Instruct
# TeleQnA
#
# 50 HELD-OUT TEST QUESTIONS
#
# IMPORTANT:
# - NO RAG
# - NO CHROMADB
# - NO MISTRAL
# - NO AGENT ROUTER
#
# This experiment measures the effect of QLoRA fine-tuning
# itself.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import json
import re
import time
import random

import torch

from pathlib import Path

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

import pandas as pd


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_MODEL = (
    r"C:\Users\ASUS\Desktop"
    r"\telecom_project\models\qwen2.5-1.5b"
)

ADAPTER_PATH = "Tuned_model"

DATA_FILE = Path(r"Fine-tuning\TeleQnA.json")

SEED = 42

NUM_TEST_QUESTIONS = 50

MAX_NEW_TOKENS = 32

OUTPUT_CSV = "base_vs_finetuned_50_results.csv"


# ============================================================
# 3. GPU
# ============================================================

print("=" * 70)
print("SYSTEM")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is required for this evaluation."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 4. LOAD TELEQnA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TELEQnA")
print("=" * 70)

with open(
    DATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(
    "Total TeleQnA questions:",
    len(data)
)


# ============================================================
# 5. CONVERT TO SAME FORMAT USED DURING TRAINING
# ============================================================

records = []

for item_id, item in data.items():

    question = item["question"]

    option_keys = [
        k
        for k in item.keys()
        if k.startswith("option")
    ]

    # Keep options in deterministic order
    option_keys = sorted(
        option_keys,
        key=lambda x: int(
            re.search(r"\d+", x).group()
        )
    )

    options = "\n".join(
        f"{i + 1}. {item[k]}"
        for i, k in enumerate(option_keys)
    )

    user_content = (
        f"Question:\n"
        f"{question}\n\n"
        f"Options:\n"
        f"{options}"
    )

    records.append({

        "id": item_id,

        "question": question,

        "options": options,

        "answer": item["answer"],

        "messages": [

            {
                "role": "user",
                "content": user_content
            },

            {
                "role": "assistant",
                "content": item["answer"]
            }

        ]
    })


print(
    "Converted examples:",
    len(records)
)


# ============================================================
# 6. RECREATE THE EXACT TRAIN / VALIDATION / TEST SPLIT
# ============================================================

print("\n" + "=" * 70)
print("RECREATING DATA SPLIT")
print("=" * 70)

dataset = Dataset.from_list(records)

dataset = dataset.shuffle(
    seed=SEED
)


# ------------------------------
# 80% TRAIN
# 20% TEMP
# ------------------------------

train_test = dataset.train_test_split(
    test_size=0.20,
    seed=SEED
)

train_dataset = train_test["train"]

temp_test_dataset = train_test["test"]


# ------------------------------
# Split remaining 20%
#
# 50% validation
# 50% test
#
# Therefore:
#
# 80% train
# 10% validation
# 10% test
# ------------------------------

validation_test = temp_test_dataset.train_test_split(
    test_size=0.50,
    seed=SEED
)

validation_dataset = validation_test["train"]

test_dataset = validation_test["test"]


print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(validation_dataset)
)

print(
    "Test:",
    len(test_dataset)
)


# ============================================================
# 7. SELECT 50 HELD-OUT TEST QUESTIONS
# ============================================================

if len(test_dataset) < NUM_TEST_QUESTIONS:

    raise ValueError(
        f"Test set only contains {len(test_dataset)} "
        f"questions."
    )


# Use a fixed random seed so the 50 questions are reproducible.

random.seed(SEED)

test_indices = random.sample(
    range(len(test_dataset)),
    NUM_TEST_QUESTIONS
)

evaluation_dataset = test_dataset.select(
    test_indices
)


print("\n" + "=" * 70)
print("FINAL EVALUATION SET")
print("=" * 70)

print(
    "Held-out questions:",
    len(evaluation_dataset)
)

print(
    "\nEvaluation IDs:"
)

for i, example in enumerate(
    evaluation_dataset
):

    print(
        f"{i + 1:02d}. {example['id']}"
    )


# ============================================================
# 8. TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 9. 4-BIT QUANTIZATION
# ============================================================

print("\n" + "=" * 70)
print("CONFIGURING 4-BIT QUANTIZATION")
print("=" * 70)

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.bfloat16,
)


# ============================================================
# 10. LOAD BASE MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING BASE QWEN MODEL")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(

    BASE_MODEL,

    quantization_config=bnb_config,

    device_map={"": 0},

    torch_dtype=torch.bfloat16,
)

base_model.eval()

print(
    "Base model loaded successfully."
)


# ============================================================
# 11. LOAD FINE-TUNED MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING FINE-TUNED QLORA MODEL")
print("=" * 70)

# Load another copy of the base model.

ft_base_model = AutoModelForCausalLM.from_pretrained(

    BASE_MODEL,

    quantization_config=bnb_config,

    device_map={"": 0},

    torch_dtype=torch.bfloat16,
)


# Load LoRA adapter on top of the base model.

fine_tuned_model = PeftModel.from_pretrained(

    ft_base_model,

    ADAPTER_PATH,
)

fine_tuned_model.eval()

print(
    "Fine-tuned model loaded successfully."
)


# ============================================================
# 12. GENERATION FUNCTION
# ============================================================

def generate_answer(
    model,
    prompt,
):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,
    )

    inputs = tokenizer(

        text,

        return_tensors="pt",

        truncation=True,

        max_length=512,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    start_time = time.perf_counter()

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            temperature=None,

            top_p=None,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=tokenizer.eos_token_id,
        )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    # Only decode newly generated tokens.

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(

        generated_tokens,

        skip_special_tokens=True,
    ).strip()

    return answer, elapsed


# ============================================================
# 13. EXTRACT OPTION NUMBER
# ============================================================

def extract_option_number(text):

    if not text:

        return None

    text = text.strip().lower()


    # --------------------------------------------------------
    # Pattern 1
    #
    # option 2
    # option: 2
    # --------------------------------------------------------

    match = re.search(
        r"\boption\s*[:\-]?\s*(\d+)\b",
        text,
    )

    if match:

        return int(
            match.group(1)
        )


    # --------------------------------------------------------
    # Pattern 2
    #
    # 2.
    # 2:
    # --------------------------------------------------------

    match = re.match(
        r"^\s*(\d+)\s*[\.\):\-]",
        text,
    )

    if match:

        return int(
            match.group(1)
        )


    # --------------------------------------------------------
    # Pattern 3
    #
    # (2)
    # --------------------------------------------------------

    match = re.search(
        r"\((\d+)\)",
        text,
    )

    if match:

        return int(
            match.group(1)
        )


    # --------------------------------------------------------
    # Pattern 4
    #
    # first / second / third / fourth
    # --------------------------------------------------------

    words = {

        "first": 1,
        "second": 2,
        "third": 3,
        "fourth": 4,
        "fifth": 5,
    }

    for word, number in words.items():

        if re.search(
            rf"\b{word}\b",
            text
        ):

            return number


    return None


# ============================================================
# 14. EXTRACT GROUND-TRUTH OPTION
# ============================================================

def get_correct_option(answer):

    option = extract_option_number(
        answer
    )

    return option


# ============================================================
# 15. EVALUATE 50 QUESTIONS
# ============================================================

print("\n" + "=" * 70)
print("STARTING BASE vs FINE-TUNED EVALUATION")
print("=" * 70)

results = []


for index, example in enumerate(
    evaluation_dataset
):

    question_number = index + 1

    question = example["question"]

    options = example["options"]

    ground_truth = example["answer"]

    correct_option = get_correct_option(
        ground_truth
    )

    prompt = (
        f"Question:\n"
        f"{question}\n\n"
        f"Options:\n"
        f"{options}\n\n"
        f"Answer the question by selecting "
        f"one option.\n"
        f"Return the option number and answer."
    )


    print("\n" + "-" * 70)

    print(
        f"QUESTION {question_number}/"
        f"{NUM_TEST_QUESTIONS}"
    )

    print(
        "Question:",
        question
    )

    print(
        "Ground truth:",
        ground_truth
    )

    print(
        "Correct option:",
        correct_option
    )


    # ========================================================
    # BASE MODEL
    # ========================================================

    print("\nBASE MODEL")

    base_answer, base_latency = (
        generate_answer(
            base_model,
            prompt,
        )
    )

    base_option = extract_option_number(
        base_answer
    )

    base_correct = (
        base_option == correct_option
    )

    print(
        "Answer:",
        base_answer
    )

    print(
        "Selected option:",
        base_option
    )

    print(
        "Correct:",
        base_correct
    )

    print(
        "Latency:",
        round(base_latency, 3),
        "seconds"
    )


    # ========================================================
    # FINE-TUNED MODEL
    # ========================================================

    print("\nFINE-TUNED MODEL")

    ft_answer, ft_latency = (
        generate_answer(
            fine_tuned_model,
            prompt,
        )
    )

    ft_option = extract_option_number(
        ft_answer
    )

    ft_correct = (
        ft_option == correct_option
    )

    print(
        "Answer:",
        ft_answer
    )

    print(
        "Selected option:",
        ft_option
    )

    print(
        "Correct:",
        ft_correct
    )

    print(
        "Latency:",
        round(ft_latency, 3),
        "seconds"
    )


    # ========================================================
    # STORE RESULT
    # ========================================================

    results.append({

        "question_id":
            example["id"],

        "question":
            question,

        "ground_truth":
            ground_truth,

        "correct_option":
            correct_option,

        "base_answer":
            base_answer,

        "base_option":
            base_option,

        "base_correct":
            base_correct,

        "base_latency_sec":
            base_latency,

        "finetuned_answer":
            ft_answer,

        "finetuned_option":
            ft_option,

        "finetuned_correct":
            ft_correct,

        "finetuned_latency_sec":
            ft_latency,
    })


# ============================================================
# 16. CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)


# ============================================================
# 17. CALCULATE METRICS
# ============================================================

base_correct_count = (
    results_df["base_correct"]
    .sum()
)

ft_correct_count = (
    results_df["finetuned_correct"]
    .sum()
)


base_accuracy = (
    base_correct_count
    / NUM_TEST_QUESTIONS
    * 100
)


ft_accuracy = (
    ft_correct_count
    / NUM_TEST_QUESTIONS
    * 100
)


accuracy_improvement = (
    ft_accuracy
    - base_accuracy
)


base_avg_latency = (
    results_df["base_latency_sec"]
    .mean()
)


ft_avg_latency = (
    results_df["finetuned_latency_sec"]
    .mean()
)


# ============================================================
# 18. PRINT FINAL REPORT
# ============================================================

print("\n\n" + "#" * 70)

print(
    "EXPERIMENT A"
)

print(
    "BASE MODEL vs FINE-TUNED MODEL"
)

print(
    "#" * 70
)


print(
    "\nEvaluation questions:",
    NUM_TEST_QUESTIONS
)


print("\n" + "=" * 70)

print("RESULTS")

print("=" * 70)


print(
    f"\nBase model:"
)

print(
    f"  Correct: {base_correct_count}/"
    f"{NUM_TEST_QUESTIONS}"
)

print(
    f"  Accuracy: {base_accuracy:.2f}%"
)

print(
    f"  Average latency: "
    f"{base_avg_latency:.3f} sec"
)


print(
    f"\nFine-tuned model:"
)

print(
    f"  Correct: {ft_correct_count}/"
    f"{NUM_TEST_QUESTIONS}"
)

print(
    f"  Accuracy: {ft_accuracy:.2f}%"
)

print(
    f"  Average latency: "
    f"{ft_avg_latency:.3f} sec"
)


print(
    "\nAccuracy improvement:"
)

print(
    f"  {accuracy_improvement:+.2f} percentage points"
)


# ============================================================
# 19. QUESTION-BY-QUESTION COMPARISON
# ============================================================

base_wins = 0

ft_wins = 0

both_correct = 0

both_wrong = 0


for _, row in results_df.iterrows():

    if (
        row["base_correct"]
        and row["finetuned_correct"]
    ):

        both_correct += 1

    elif (
        not row["base_correct"]
        and not row["finetuned_correct"]
    ):

        both_wrong += 1

    elif row["finetuned_correct"]:

        ft_wins += 1

    elif row["base_correct"]:

        base_wins += 1


print("\n" + "=" * 70)

print("QUESTION-LEVEL COMPARISON")

print("=" * 70)


print(
    "\nBoth correct:",
    both_correct
)

print(
    "Both wrong:",
    both_wrong
)

print(
    "Fine-tuned correct / Base wrong:",
    ft_wins
)

print(
    "Base correct / Fine-tuned wrong:",
    base_wins
)


# ============================================================
# 20. SAVE RESULTS
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)


print("\n" + "=" * 70)

print(
    "RESULTS SAVED"
)

print("=" * 70)

print(
    "File:",
    OUTPUT_CSV
)


# ============================================================
# 21. FINAL SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({

    "Metric": [

        "Test questions",

        "Correct answers",

        "Accuracy",

        "Average latency (sec)",
    ],

    "Base Qwen": [

        NUM_TEST_QUESTIONS,

        base_correct_count,

        f"{base_accuracy:.2f}%",

        f"{base_avg_latency:.3f}",
    ],

    "Fine-tuned Qwen": [

        NUM_TEST_QUESTIONS,

        ft_correct_count,

        f"{ft_accuracy:.2f}%",

        f"{ft_avg_latency:.3f}",
    ],
})


print("\n" + "=" * 70)

print(
    "FINAL COMPARISON TABLE"
)

print("=" * 70)

print(
    summary_df.to_string(
        index=False
    )
)


print("\nEvaluation complete.")

SYSTEM
PyTorch: 2.8.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.0 GB

LOADING TELEQnA
Total TeleQnA questions: 10000
Converted examples: 10000

RECREATING DATA SPLIT
Train: 8000
Validation: 1000
Test: 1000

FINAL EVALUATION SET
Held-out questions: 50

Evaluation IDs:
01. question 6901
02. question 144
03. question 501
04. question 3535
05. question 6002
06. question 6151
07. question 951
08. question 7630
09. question 1920
10. question 6299
11. question 9769
12. question 3152
13. question 8763
14. question 2891
15. question 7403
16. question 405
17. question 3667
18. question 9170
19. question 7266
20. question 2806
21. question 7507
22. question 491
23. question 6291
24. question 2603
25. question 3270
26. question 9192
27. question 5682
28. question 2437
29. question 6108
30. question 6629
31. question 8735
32. question 7293
33. question 2087
34. question 4571
35. question 4745
36. question 710
37. question 7486
38. question 3349
39. question 2112
40.

c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\peft\config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'kasa_config', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Fine-tuned model loaded successfully.

STARTING BASE vs FINE-TUNED EVALUATION

----------------------------------------------------------------------
QUESTION 1/50
Question: When can the UE trigger reporting of its RLM and BFD relaxation status? [3GPP Release 17]
Ground truth: option 1: When the RLM and BFD relaxation status changes
Correct option: 1

BASE MODEL
Answer: 1. When the RLM and BFD relaxation status changes
Selected option: 1
Correct: True
Latency: 1.526 seconds

FINE-TUNED MODEL
Answer: option 1: When the RLM and BFD relaxation status changes
Selected option: 1
Correct: True
Latency: 3.029 seconds

----------------------------------------------------------------------
QUESTION 2/50
Question: Why is analog over-the-air computing considered to improve privacy protection in network management?
Ground truth: option 5: It imposes random perturbations to the uploaded parameters, making it impossible to recover original information
Correct option: 5

BASE MODEL
Answer: 5.
Selecte

In [7]:
# ============================================================
# EXPERIMENT A
# BASE MODEL vs FINE-TUNED MODEL
#
# Qwen2.5-1.5B-Instruct
# TeleQnA
#
# 50 HELD-OUT TEST QUESTIONS
#
# IMPORTANT:
# - NO RAG
# - NO CHROMADB
# - NO MISTRAL
# - NO AGENT ROUTER
#
# This experiment measures the effect of QLoRA fine-tuning
# itself.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import json
import re
import time
import random

import torch

from pathlib import Path

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

import pandas as pd


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_MODEL = (
    r"C:\Users\ASUS\Desktop"
    r"\telecom_project\models\qwen2.5-1.5b"
)

ADAPTER_PATH = "Tuned_model"

DATA_FILE = Path(r"Fine-tuning\TeleQnA.json")

SEED = 42

NUM_TEST_QUESTIONS = 1000

MAX_NEW_TOKENS = 32

OUTPUT_CSV = "base_vs_finetuned_1000_results.csv"


# ============================================================
# 3. GPU
# ============================================================

print("=" * 70)
print("SYSTEM")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is required for this evaluation."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# 4. LOAD TELEQnA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TELEQnA")
print("=" * 70)

with open(
    DATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(
    "Total TeleQnA questions:",
    len(data)
)


# ============================================================
# 5. CONVERT TO SAME FORMAT USED DURING TRAINING
# ============================================================

records = []

for item_id, item in data.items():

    question = item["question"]

    option_keys = [
        k
        for k in item.keys()
        if k.startswith("option")
    ]

    # Keep options in deterministic order
    option_keys = sorted(
        option_keys,
        key=lambda x: int(
            re.search(r"\d+", x).group()
        )
    )

    options = "\n".join(
        f"{i + 1}. {item[k]}"
        for i, k in enumerate(option_keys)
    )

    user_content = (
        f"Question:\n"
        f"{question}\n\n"
        f"Options:\n"
        f"{options}"
    )

    records.append({

        "id": item_id,

        "question": question,

        "options": options,

        "answer": item["answer"],

        "messages": [

            {
                "role": "user",
                "content": user_content
            },

            {
                "role": "assistant",
                "content": item["answer"]
            }

        ]
    })


print(
    "Converted examples:",
    len(records)
)


# ============================================================
# 6. RECREATE THE EXACT TRAIN / VALIDATION / TEST SPLIT
# ============================================================

print("\n" + "=" * 70)
print("RECREATING DATA SPLIT")
print("=" * 70)

dataset = Dataset.from_list(records)

dataset = dataset.shuffle(
    seed=SEED
)


# ------------------------------
# 80% TRAIN
# 20% TEMP
# ------------------------------

train_test = dataset.train_test_split(
    test_size=0.20,
    seed=SEED
)

train_dataset = train_test["train"]

temp_test_dataset = train_test["test"]


# ------------------------------
# Split remaining 20%
#
# 50% validation
# 50% test
#
# Therefore:
#
# 80% train
# 10% validation
# 10% test
# ------------------------------

validation_test = temp_test_dataset.train_test_split(
    test_size=0.50,
    seed=SEED
)

validation_dataset = validation_test["train"]

test_dataset = validation_test["test"]


print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(validation_dataset)
)

print(
    "Test:",
    len(test_dataset)
)


# ============================================================
# 7. SELECT 50 HELD-OUT TEST QUESTIONS
# ============================================================

if len(test_dataset) < NUM_TEST_QUESTIONS:

    raise ValueError(
        f"Test set only contains {len(test_dataset)} "
        f"questions."
    )


# Use a fixed random seed so the 50 questions are reproducible.

random.seed(SEED)

test_indices = random.sample(
    range(len(test_dataset)),
    NUM_TEST_QUESTIONS
)

evaluation_dataset = test_dataset.select(
    test_indices
)


print("\n" + "=" * 70)
print("FINAL EVALUATION SET")
print("=" * 70)

print(
    "Held-out questions:",
    len(evaluation_dataset)
)

print(
    "\nEvaluation IDs:"
)

for i, example in enumerate(
    evaluation_dataset
):

    print(
        f"{i + 1:02d}. {example['id']}"
    )


# ============================================================
# 8. TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# 9. 4-BIT QUANTIZATION
# ============================================================

print("\n" + "=" * 70)
print("CONFIGURING 4-BIT QUANTIZATION")
print("=" * 70)

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=torch.bfloat16,
)


# ============================================================
# 10. LOAD BASE MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING BASE QWEN MODEL")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(

    BASE_MODEL,

    quantization_config=bnb_config,

    device_map={"": 0},

    torch_dtype=torch.bfloat16,
)

base_model.eval()

print(
    "Base model loaded successfully."
)


# ============================================================
# 11. LOAD FINE-TUNED MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING FINE-TUNED QLORA MODEL")
print("=" * 70)

# Load another copy of the base model.

ft_base_model = AutoModelForCausalLM.from_pretrained(

    BASE_MODEL,

    quantization_config=bnb_config,

    device_map={"": 0},

    torch_dtype=torch.bfloat16,
)


# Load LoRA adapter on top of the base model.

fine_tuned_model = PeftModel.from_pretrained(

    ft_base_model,

    ADAPTER_PATH,
)

fine_tuned_model.eval()

print(
    "Fine-tuned model loaded successfully."
)


# ============================================================
# 12. GENERATION FUNCTION
# ============================================================

def generate_answer(
    model,
    prompt,
):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True,
    )

    inputs = tokenizer(

        text,

        return_tensors="pt",

        truncation=True,

        max_length=512,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    start_time = time.perf_counter()

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            temperature=None,

            top_p=None,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=tokenizer.eos_token_id,
        )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    # Only decode newly generated tokens.

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(

        generated_tokens,

        skip_special_tokens=True,
    ).strip()

    return answer, elapsed


# ============================================================
# 13. EXTRACT OPTION NUMBER
# ============================================================

def extract_option_number(text):

    if not text:

        return None

    text = text.strip().lower()


    # --------------------------------------------------------
    # Pattern 1
    #
    # option 2
    # option: 2
    # --------------------------------------------------------

    match = re.search(
        r"\boption\s*[:\-]?\s*(\d+)\b",
        text,
    )

    if match:

        return int(
            match.group(1)
        )


    # --------------------------------------------------------
    # Pattern 2
    #
    # 2.
    # 2:
    # --------------------------------------------------------

    match = re.match(
        r"^\s*(\d+)\s*[\.\):\-]",
        text,
    )

    if match:

        return int(
            match.group(1)
        )


    # --------------------------------------------------------
    # Pattern 3
    #
    # (2)
    # --------------------------------------------------------

    match = re.search(
        r"\((\d+)\)",
        text,
    )

    if match:

        return int(
            match.group(1)
        )


    # --------------------------------------------------------
    # Pattern 4
    #
    # first / second / third / fourth
    # --------------------------------------------------------

    words = {

        "first": 1,
        "second": 2,
        "third": 3,
        "fourth": 4,
        "fifth": 5,
    }

    for word, number in words.items():

        if re.search(
            rf"\b{word}\b",
            text
        ):

            return number


    return None


# ============================================================
# 14. EXTRACT GROUND-TRUTH OPTION
# ============================================================

def get_correct_option(answer):

    option = extract_option_number(
        answer
    )

    return option


# ============================================================
# 15. EVALUATE 50 QUESTIONS
# ============================================================

print("\n" + "=" * 70)
print("STARTING BASE vs FINE-TUNED EVALUATION")
print("=" * 70)

results = []


for index, example in enumerate(
    evaluation_dataset
):

    question_number = index + 1

    question = example["question"]

    options = example["options"]

    ground_truth = example["answer"]

    correct_option = get_correct_option(
        ground_truth
    )

    prompt = (
        f"Question:\n"
        f"{question}\n\n"
        f"Options:\n"
        f"{options}\n\n"
        f"Answer the question by selecting "
        f"one option.\n"
        f"Return the option number and answer."
    )


    print("\n" + "-" * 70)

    print(
        f"QUESTION {question_number}/"
        f"{NUM_TEST_QUESTIONS}"
    )

    print(
        "Question:",
        question
    )

    print(
        "Ground truth:",
        ground_truth
    )

    print(
        "Correct option:",
        correct_option
    )


    # ========================================================
    # BASE MODEL
    # ========================================================

    print("\nBASE MODEL")

    base_answer, base_latency = (
        generate_answer(
            base_model,
            prompt,
        )
    )

    base_option = extract_option_number(
        base_answer
    )

    base_correct = (
        base_option == correct_option
    )

    print(
        "Answer:",
        base_answer
    )

    print(
        "Selected option:",
        base_option
    )

    print(
        "Correct:",
        base_correct
    )

    print(
        "Latency:",
        round(base_latency, 3),
        "seconds"
    )


    # ========================================================
    # FINE-TUNED MODEL
    # ========================================================

    print("\nFINE-TUNED MODEL")

    ft_answer, ft_latency = (
        generate_answer(
            fine_tuned_model,
            prompt,
        )
    )

    ft_option = extract_option_number(
        ft_answer
    )

    ft_correct = (
        ft_option == correct_option
    )

    print(
        "Answer:",
        ft_answer
    )

    print(
        "Selected option:",
        ft_option
    )

    print(
        "Correct:",
        ft_correct
    )

    print(
        "Latency:",
        round(ft_latency, 3),
        "seconds"
    )


    # ========================================================
    # STORE RESULT
    # ========================================================

    results.append({

        "question_id":
            example["id"],

        "question":
            question,

        "ground_truth":
            ground_truth,

        "correct_option":
            correct_option,

        "base_answer":
            base_answer,

        "base_option":
            base_option,

        "base_correct":
            base_correct,

        "base_latency_sec":
            base_latency,

        "finetuned_answer":
            ft_answer,

        "finetuned_option":
            ft_option,

        "finetuned_correct":
            ft_correct,

        "finetuned_latency_sec":
            ft_latency,
    })


# ============================================================
# 16. CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)


# ============================================================
# 17. CALCULATE METRICS
# ============================================================

base_correct_count = (
    results_df["base_correct"]
    .sum()
)

ft_correct_count = (
    results_df["finetuned_correct"]
    .sum()
)


base_accuracy = (
    base_correct_count
    / NUM_TEST_QUESTIONS
    * 100
)


ft_accuracy = (
    ft_correct_count
    / NUM_TEST_QUESTIONS
    * 100
)


accuracy_improvement = (
    ft_accuracy
    - base_accuracy
)


base_avg_latency = (
    results_df["base_latency_sec"]
    .mean()
)


ft_avg_latency = (
    results_df["finetuned_latency_sec"]
    .mean()
)


# ============================================================
# 18. PRINT FINAL REPORT
# ============================================================

print("\n\n" + "#" * 70)

print(
    "EXPERIMENT A"
)

print(
    "BASE MODEL vs FINE-TUNED MODEL"
)

print(
    "#" * 70
)


print(
    "\nEvaluation questions:",
    NUM_TEST_QUESTIONS
)


print("\n" + "=" * 70)

print("RESULTS")

print("=" * 70)


print(
    f"\nBase model:"
)

print(
    f"  Correct: {base_correct_count}/"
    f"{NUM_TEST_QUESTIONS}"
)

print(
    f"  Accuracy: {base_accuracy:.2f}%"
)

print(
    f"  Average latency: "
    f"{base_avg_latency:.3f} sec"
)


print(
    f"\nFine-tuned model:"
)

print(
    f"  Correct: {ft_correct_count}/"
    f"{NUM_TEST_QUESTIONS}"
)

print(
    f"  Accuracy: {ft_accuracy:.2f}%"
)

print(
    f"  Average latency: "
    f"{ft_avg_latency:.3f} sec"
)


print(
    "\nAccuracy improvement:"
)

print(
    f"  {accuracy_improvement:+.2f} percentage points"
)


# ============================================================
# 19. QUESTION-BY-QUESTION COMPARISON
# ============================================================

base_wins = 0

ft_wins = 0

both_correct = 0

both_wrong = 0


for _, row in results_df.iterrows():

    if (
        row["base_correct"]
        and row["finetuned_correct"]
    ):

        both_correct += 1

    elif (
        not row["base_correct"]
        and not row["finetuned_correct"]
    ):

        both_wrong += 1

    elif row["finetuned_correct"]:

        ft_wins += 1

    elif row["base_correct"]:

        base_wins += 1


print("\n" + "=" * 70)

print("QUESTION-LEVEL COMPARISON")

print("=" * 70)


print(
    "\nBoth correct:",
    both_correct
)

print(
    "Both wrong:",
    both_wrong
)

print(
    "Fine-tuned correct / Base wrong:",
    ft_wins
)

print(
    "Base correct / Fine-tuned wrong:",
    base_wins
)


# ============================================================
# 20. SAVE RESULTS
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)


print("\n" + "=" * 70)

print(
    "RESULTS SAVED"
)

print("=" * 70)

print(
    "File:",
    OUTPUT_CSV
)


# ============================================================
# 21. FINAL SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({

    "Metric": [

        "Test questions",

        "Correct answers",

        "Accuracy",

        "Average latency (sec)",
    ],

    "Base Qwen": [

        NUM_TEST_QUESTIONS,

        base_correct_count,

        f"{base_accuracy:.2f}%",

        f"{base_avg_latency:.3f}",
    ],

    "Fine-tuned Qwen": [

        NUM_TEST_QUESTIONS,

        ft_correct_count,

        f"{ft_accuracy:.2f}%",

        f"{ft_avg_latency:.3f}",
    ],
})


print("\n" + "=" * 70)

print(
    "FINAL COMPARISON TABLE"
)

print("=" * 70)

print(
    summary_df.to_string(
        index=False
    )
)


print("\nEvaluation complete.")

SYSTEM
PyTorch: 2.8.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.0 GB

LOADING TELEQnA
Total TeleQnA questions: 10000
Converted examples: 10000

RECREATING DATA SPLIT
Train: 8000
Validation: 1000
Test: 1000

FINAL EVALUATION SET
Held-out questions: 1000

Evaluation IDs:
01. question 6901
02. question 144
03. question 501
04. question 3535
05. question 6002
06. question 6151
07. question 951
08. question 7630
09. question 1920
10. question 6299
11. question 9769
12. question 3152
13. question 8763
14. question 2891
15. question 7403
16. question 405
17. question 3667
18. question 9170
19. question 7266
20. question 2806
21. question 7507
22. question 491
23. question 6291
24. question 2603
25. question 3270
26. question 9192
27. question 5682
28. question 2437
29. question 6108
30. question 6629
31. question 1292
32. question 8735
33. question 7293
34. question 2087
35. question 4571
36. question 4745
37. question 710
38. question 7486
39. question 3349
4

c:\Users\ASUS\miniconda3\envs\tf\lib\site-packages\peft\config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'kasa_config', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Fine-tuned model loaded successfully.

STARTING BASE vs FINE-TUNED EVALUATION

----------------------------------------------------------------------
QUESTION 1/1000
Question: When can the UE trigger reporting of its RLM and BFD relaxation status? [3GPP Release 17]
Ground truth: option 1: When the RLM and BFD relaxation status changes
Correct option: 1

BASE MODEL
Answer: 1. When the RLM and BFD relaxation status changes
Selected option: 1
Correct: True
Latency: 2.512 seconds

FINE-TUNED MODEL
Answer: option 1: When the RLM and BFD relaxation status changes
Selected option: 1
Correct: True
Latency: 2.412 seconds

----------------------------------------------------------------------
QUESTION 2/1000
Question: Why is analog over-the-air computing considered to improve privacy protection in network management?
Ground truth: option 5: It imposes random perturbations to the uploaded parameters, making it impossible to recover original information
Correct option: 5

BASE MODEL
Answer: 5.
Sel

In [8]:
import pandas as pd
df = pd.read_csv("base_vs_finetuned_1000_results.csv")
df.head()

,question_id,question,ground_truth,correct_option,base_answer,base_option,base_correct,base_latency_sec,finetuned_answer,finetuned_option,finetuned_correct,finetuned_latency_sec
0,question 6901,When can the UE trigger reporting of its RLM a...,option 1: When the RLM and BFD relaxation stat...,1,1. When the RLM and BFD relaxation status changes,1.0,True,2.512476,option 1: When the RLM and BFD relaxation stat...,1,True,2.412474
1,question 144,Why is analog over-the-air computing considere...,option 5: It imposes random perturbations to t...,5,5.,5.0,True,1.159790,option 5: It imposes random perturbations to t...,5,True,3.479443
2,question 501,What is the term for a network where every swi...,option 2: Full-mesh network,2,2. Full-mesh network,2.0,True,1.450386,option 2: Full-mesh network,2,True,1.614627
3,question 3535,In which type of terrain cover are AG (air to ...,option 1: Open space,1,1. 2. Hilly/Mountainous,1.0,True,1.908094,option 2: Hilly/Mountainous,2,False,1.713983
4,question 6002,What are the three classes into which existing...,"option 2: Throughput fairness scheduling, thro...",2,"1. Throughput fairness scheduling, throughput ...",1.0,False,3.054839,"option 2: Throughput fairness scheduling, thro...",2,True,3.060809
